# ML Regression Algorithms from Scratch - Multiple Linear Regression 

## Introduction

In this notebook, first, we implement Multiple Linear Regression from Scratch using Numpy without Sklearn. After the scratch implementation, we also implement the Multiple Linear Regression using Sklearn and compare the two models. The complete code is written and executed in Google Colab. No need of installing any additional packages is required. Download this notebook and upload to Google Colab to run it by yourself. Don't forget to grab your Datasets! 

The Dataset used here is the **50_Startups** dataset provided by the Super Data Science Team under their programme **Machine Learning A-Z: Hands on Python & R in Data Science**. Find the various datasets provided by them [here](https://www.superdatascience.com/pages/machine-learning).

**Response Variable** : R&D Spend, Administration, Marketing Spend, State

**Target Variable** : Profit

**Equation Used** : y = W0 * X0 + W1 * X1 + W2 * X2 + .... + WN * XN

**For Scratch Implementation:**
    
    Loss Function : Mean Squared Error

    Optimization Algorithm : SGD

    Weight Initialization : Xavier Initialization



## Multiple Linear Regression from Scratch without SKlearn

### Importing the Dependencies

In [ ]:
#importing the required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Get the Data and Data Preprocessing

In [ ]:
#Get the dataset
dataset = pd.read_csv(r'../input/50-startups-by-superdatascience-team/50_Startups.csv')

In [ ]:
#Get a glimpse of the Dataset
dataset.head()

In [ ]:
#Separating the independent and dependent features
#Dependent features
y = np.asarray(dataset['Profit'].values.tolist()) 

# Independent Features
# Now, our dataset has only independent features
dataset.drop(["Profit"], axis = 1, inplace = True)

#### Handling the Categorical Variable "**State**"

In [ ]:
# Handling the Categorical Variable "State" with the One Hot Encoding Technique
# There are many ways of handling categorical variables, One Hot Encosing is one of them
# First, we get the counts of each value that the feature "State" can take
dataset.iloc[:,3].value_counts()

In [ ]:
# Performing Label Encoding
# Replacing California by 1, New York by 2, Florida by 3
dataset.replace(to_replace=["California","New York", "Florida"], value=[1,2,3])

In [ ]:
# There is no hierarchy relation between the labels
# i.e label 3 does not have more importance than label 1
# To handle this, we create 3 columns with the respective US State names
dataset["California"] = dataset.iloc[:, 3]
dataset["New York"] = dataset.iloc[:,3]
dataset["Florida"] = dataset.iloc[:,3]

In [ ]:
#Let's have a look at the dataset now 
dataset.head()

Now, we replace the values of the particular US State of the particular US State Column by one and others by zero. This means, in the column California, we will replace all the California Values by 1 and the values New York, Florida by zero. This is called **One Hot Encoding**. We do this for each of the three columns. After this, we can drop our original column "State".

Also, note that to avoid "**Dummy Variable Trap**", we need to drop one of the three US State columns. Basically, Linear Regression has an assumption that all the Independent Variables(X), in the equation **y = W0 * X0 + W1 * X1 + W2 * X2 + ... WN * XN**, are independent of each other. If we don't drop one of the three columns, we violate this assumption.

In [ ]:
# Performing One Hot Encoding for the column "California"
dataset.loc[dataset["California"]!="California", "California"] = 0
dataset.loc[dataset["California"]=="California", "California"] = 1

In [ ]:
# Performing One Hot Encoding for the column "New York"
dataset.loc[dataset["New York"]!="New York", "New York"] = 0
dataset.loc[dataset["New York"]=="New York", "New York"] = 1

In [ ]:
# Performing One Hot Encoding for the column "Florida"
dataset.loc[dataset.Florida!="Florida", "Florida"] = 0
dataset.loc[dataset.Florida=="Florida", "Florida"] = 1

In [ ]:
#Let's have a look at the Data
dataset.head()

By looking at the US State columns and the "State" column, we can clearly see that we have successfully performed one hot encoding. Now, it's time to drop the "State" column and one of the three State columns. We will drop the column "Florida".

In [ ]:
# Dropping the columns
dataset.drop(["State","Florida"], axis = 1, inplace = True)

In [ ]:
dataset.head()

### Data Preprocessing Continued

In [ ]:
# Get the processed Independent features 
X = np.asarray(dataset.values.tolist())

In [ ]:
#Get the shapes of X and y
print("The shape of the independent fatures are ",X.shape)
print("The shape of the dependent fatures are ",y.shape)

In [ ]:
#Reshaping the Dependent features
y = y.reshape(len(y),1) # Changing the shape from (50,) to (50,1)

In [ ]:
#Feature Scaling for Independent Variables
for i in range(X.shape[1]-2):
  X[:,i] = (X[:,i] - int(np.mean(X[:,i])))/np.std(X[:,i])


In [ ]:
#Feature Scaling for Dependent Variables
y = (y - int(np.mean(y)))/np.std(y)

In [ ]:
#Adding the feature X0 = 1, so we have the equation: y =  (W1 * X1) + (W0 * X0) 
X = np.concatenate((X,np.ones((50,1))), axis = 1)

In [ ]:
X

In [ ]:
y

In [ ]:
#Let's create a DataFrame "Independent_Variables" to visualize our final independent features
Indpendent_Variables = pd.DataFrame(X)

In [ ]:
Indpendent_Variables

### Utility Methods

In [ ]:
# The method "split_data" splits the given dataset into trainset and testset
# This is similar to the method "train_test_split" from "sklearn.model_selection"
def split_data(X,y,test_size=0.2,random_state=0):
    np.random.seed(random_state)                  #set the seed for reproducible results
    indices = np.random.permutation(len(X))       #shuffling the indices
    data_test_size = int(X.shape[0] * test_size)  #Get the test size

    #Separating the Independent and Dependent features into the Train and Test Set
    train_indices = indices[data_test_size:]
    test_indices = indices[:data_test_size]
    X_train = X[train_indices]
    y_train = y[train_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]
    return X_train, y_train, X_test, y_test

### Coding the LinearRegression Class

In [ ]:
class multipleLinearRegression():

  def __init__(self):
    #No instance Variables required
    pass

  def forward(self,X,y,W):
    """
    Parameters:
    X (array) : Independent Features
    y (array) : Dependent Features/ Target Variable
    W (array) : Weights 

    Returns:
    loss (float) : Calculated Sqaured Error Loss for y and y_pred
    y_pred (array) : Predicted Target Variable
    """
    y_pred = sum(W * X)
    loss = ((y_pred-y)**2)/2    #Loss = Squared Error, we introduce 1/2 for ease in the calculation
    return loss, y_pred

  def updateWeights(self,X,y_pred,y_true,W,alpha,index):
    """
    Parameters:
    X (array) : Independent Features
    y_pred (array) : Predicted Target Variable
    y_true (array) : Dependent Features/ Target Variable
    W (array) : Weights
    alpha (float) : learning rate
    index (int) : Index to fetch the corresponding values of W, X and y 

    Returns:
    W (array) : Update Values of Weight
    """
    for i in range(X.shape[1]):
      #alpha = learning rate, rest of the RHS is derivative of loss function
      W[i] -= (alpha * (y_pred-y_true[index])*X[index][i]) 
    return W

  def train(self, X, y, epochs=10, alpha=0.001, random_state=0):
    """
    Parameters:
    X (array) : Independent Feature
    y (array) : Dependent Features/ Target Variable
    epochs (int) : Number of epochs for training, default value is 10
    alpha (float) : learning rate, default value is 0.001

    Returns:
    y_pred (array) : Predicted Target Variable
    loss (float) : Calculated Sqaured Error Loss for y and y_pred
    """

    num_rows = X.shape[0] #Number of Rows
    num_cols = X.shape[1] #Number of Columns 
    W = np.random.randn(1,num_cols) / np.sqrt(num_rows) #Weight Initialization

    #Calculating Loss and Updating Weights
    train_loss = []
    num_epochs = []
    train_indices = [i for i in range(X.shape[0])]
    for j in range(epochs):
      cost=0
      np.random.seed(random_state)
      np.random.shuffle(train_indices)
      for i in train_indices:
        loss, y_pred = self.forward(X[i],y[i],W[0])
        cost+=loss
        W[0] = self.updateWeights(X,y_pred,y,W[0],alpha,i)
      train_loss.append(cost)
      num_epochs.append(j)
    return W[0], train_loss, num_epochs

  def test(self, X_test, y_test, W_trained):
    """
    Parameters:
    X_test (array) : Independent Features from the Test Set
    y_test (array) : Dependent Features/ Target Variable from the Test Set
    W_trained (array) : Trained Weights
    test_indices (list) : Index to fetch the corresponding values of W_trained,
                          X_test and y_test 

    Returns:
    test_pred (list) : Predicted Target Variable
    test_loss (list) : Calculated Sqaured Error Loss for y and y_pred
    """
    test_pred = []
    test_loss = []
    test_indices = [i for i in range(X_test.shape[0])]
    for i in test_indices:
        loss, y_test_pred = self.forward(X_test[i], W_trained, y_test[i])
        test_pred.append(y_test_pred)
        test_loss.append(loss)
    return test_pred, test_loss
    

  def predict(self, W_trained, X_sample):
    prediction = sum(W_trained * X_sample)
    return prediction

  def plotLoss(self, loss, epochs):
    """
    Parameters:
    loss (list) : Calculated Sqaured Error Loss for y and y_pred
    epochs (list): Number of Epochs

    Returns: None
    Plots a graph of Loss vs Epochs
    """
    plt.plot(epochs, loss)
    plt.xlabel('Number of Epochs')
    plt.ylabel('Loss')
    plt.title('Plot Loss')
    plt.show()
  


### Performing Linear Regression

In [ ]:
#Splitting the dataset
X_train, y_train, X_test, y_test = split_data(X,y)

In [ ]:
#declaring the "regressor" as an object of the class LinearRegression
regressor = multipleLinearRegression()

In [ ]:
#Training 
W_trained, train_loss, num_epochs = regressor.train(X_train, y_train, epochs=200, alpha=0.0001)

In [ ]:
#Testing on the Test Dataset
test_pred, test_loss = regressor.test(X_test, y_test, W_trained)

### Visualizing Results

In [ ]:
#Plot the Train Loss
regressor.plotLoss(train_loss, num_epochs)

## Multople Linear Regression using Sklearn

### Importing dependencies

In [ ]:
import pandas
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

### Data Preprocessing

In [ ]:
#Get the Dataset and separating the independent and dependent features
dataset_sk = pd.read_csv(r'../input/50-startups-by-superdatascience-team/50_Startups.csv')
X_sk = dataset_sk.iloc[:, :-1].values
y_sk = dataset_sk.iloc[:, 4].values

In [ ]:
# Performing One hot encoding using sklearn
labelencoder_X_sk = LabelEncoder()
X_sk[:,3] = labelencoder_X_sk.fit_transform(X_sk[:,3])

onehotencoder = OneHotEncoder(handle_unknown='ignore')
X_sk_categorical = onehotencoder.fit_transform(X_sk[:,3].reshape(-1,1)).toarray()
X_sk = np.concatenate((X_sk,X_sk_categorical),axis=1)

In [ ]:
# Dropping the "State" and "California" columns
X_sk = X_sk[:, [0,1,2,5,6]]

In [ ]:
X_sk.shape

### Performing the Linear Regression

In [ ]:
# Splitting the dataset into the Training set and Test set
X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(X_sk, y_sk, test_size = 0.2, random_state = 0)

In [ ]:
# Fitting Simple Linear Regression to the Training set
regressor_sk = LinearRegression()
regressor_sk.fit(X_train_sk, y_train_sk)

In [ ]:
# Predicting the Test set results
y_pred = regressor_sk.predict(X_test_sk)

In [ ]:
X_train_sk.shape

## Comparing the Performance of the Two Regressors

In [ ]:
#Making the Prediction using Sklearn Regression
print(regressor_sk.predict([[160000,140000,5000000,1,0]]))

In [ ]:
#Making a Prediction
pred = regressor.predict(W_trained,[160000,140000,5000000,1,0,1])
print(pred)

The values used for Independent features resemble closely to the first set of features of our Dataset. Therefore, the prediction should be somewhere around 2,00,000. Therefore, SKlearn Regressor predicts better than our Naive Regressor.

There can be various reasons behind these results like choice of loss function, Optimization Algorithm used, etc. However, the prime goal of this notebook was to demonstrate the implementation of Multiple Linear Regression and not to perform better than Sklearn. 

Definitely, you can download this notebook and change hyperparameters, Optimization Algorithm, etc. and start your ML Journey!

**Best of Luck!**

